# LLM-WITHMEM: Level 4 Multi-Bank Memory Encoder Training

Train a **51.3M-parameter** multi-bank query-conditioned memory encoder on **A100 40GB**.

**Architecture**:
- Profile → SharedEncoder (4-layer) → 5 PerceiverBankHeads (episodic/semantic/procedural/emotional/prospective)
- Query → QueryEncoder (2-layer) → attention pool → query_vector
- WorkingMemory: query-conditioned cross-attention over all 72 bank slots → 32 output vectors
- DynamicGateNetwork: query → per-layer per-head sigmoid gates via MLP
- 4 layer-group K,V projections × gates → inject into SmolLM2-1.7B DynamicCache

**Training**: KL Distillation — gold prompt has ONLY query-relevant memory types, so the encoder learns which banks to activate per query.

**A100 40GB Config**: batch_size=4, grad_accum=2, effective batch=8, bf16/fp16 mixed

## 1. GPU Check & A100 Auto-Config

In [ ]:
import torch
import subprocess

assert torch.cuda.is_available(), "No GPU detected — switch to a GPU runtime!"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# ═══ Auto-select batch size based on GPU VRAM ═══
if vram_gb >= 70:     # H100 / A100 80GB
    BATCH_SIZE = 8
    GRAD_ACCUM = 1
elif vram_gb >= 38:   # A100 40GB ← your target
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
elif vram_gb >= 20:   # A10G / RTX 4090
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
elif vram_gb >= 14:   # T4 / V100
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
else:
    BATCH_SIZE = 1
    GRAD_ACCUM = 8

print(f"\n✓ Auto-config: batch_size={BATCH_SIZE}, grad_accum={GRAD_ACCUM}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

# Enable TF32 for A100 matmul speedup
if "A100" in gpu_name or "H100" in gpu_name:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("  ✓ TF32 enabled for A100/H100")

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/Pkansagra-hub/LLM-WITHMEM.git"
REPO_DIR = "/content/LLM-WITHMEM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls experiments/level4_multibank/

## 3. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate matplotlib

## 4. Generate Synthetic Training Data

Generates 200 profiles × 30 query templates = 6,000 training samples across 5 memory types (episodic, semantic, preferences, emotional, procedural).

In [ ]:
DATA_DIR = "experiments/level4_multibank/data"

!python -m experiments.level4_multibank.data.generate_profiles \
    --num-train 2000 \
    --num-val 400 \
    --output-dir {DATA_DIR}

# Quick sanity check
import json
with open(f"{DATA_DIR}/train.json") as f:
    train = json.load(f)
with open(f"{DATA_DIR}/val.json") as f:
    val = json.load(f)

print(f"\nTrain: {len(train)} examples")
print(f"Val:   {len(val)} examples")

# Show sample structure
sample = train[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Profile ({len(sample['profile_text'])} chars):")
print(sample["profile_text"][:200] + "...")
print(f"\nQuery: {sample['query_text']}")
print(f"Relevant types: {sample.get('relevant_types', 'N/A')}")
print(f"Relevant facts: {len(sample.get('relevant_facts', []))} facts")

## 5. Configure Training (A100 Tuned)

Override default config for A100 40GB: larger batch, moderate accumulation, 10k steps.

In [ ]:
import sys
sys.path.insert(0, "/content/LLM-WITHMEM")

from experiments.level4_multibank.config import Config

config = Config()

# ─── A100-tuned overrides ───
config.training.batch_size = BATCH_SIZE         # 4 on A100 40GB
config.training.gradient_accumulation_steps = GRAD_ACCUM  # 2 → effective batch 8
config.training.learning_rate = 1e-4
config.training.max_steps = 10000
config.training.eval_every = 500
config.training.save_every = 1000
config.training.warmup_steps = 200
config.training.max_new_tokens = 128
config.training.num_workers = 2                 # Colab has 2 CPUs

config.output_dir = "outputs/level4_multibank"
config.log_dir = "logs/level4_multibank"

print("Training Config:")
print(f"  batch_size:     {config.training.batch_size}")
print(f"  grad_accum:     {config.training.gradient_accumulation_steps}")
print(f"  effective_bs:   {config.training.batch_size * config.training.gradient_accumulation_steps}")
print(f"  lr:             {config.training.learning_rate}")
print(f"  max_steps:      {config.training.max_steps}")
print(f"  eval_every:     {config.training.eval_every}")
print(f"  warmup:         {config.training.warmup_steps}")
print(f"\nEncoder Config:")
print(f"  d_model:        {config.encoder.d_model}")
print(f"  n_heads:        {config.encoder.n_heads}")
print(f"  output_slots:   {config.encoder.num_output_slots}")
print(f"  layer_groups:   {config.encoder.num_layer_groups}")
print(f"  banks:          {config.banks.bank_names}")
print(f"  total_slots:    {config.banks.total_slots}")

## 6. Load Model & Create Trainer

Load SmolLM2-1.7B-Instruct (fp16), freeze it, then build the 51.3M-param multi-bank encoder.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from experiments.level4_multibank.training.trainer import Trainer

# Load frozen LLM
print("Loading SmolLM2-1.7B-Instruct (fp16)...")
tokenizer = AutoTokenizer.from_pretrained(config.model.model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.model.model_id,
    torch_dtype=torch.float16,
    attn_implementation="eager",  # needed for attention weight access
).to(config.device)

print(f"  LLM loaded: {sum(p.numel() for p in model.parameters()):,} params")
print(f"  VRAM after LLM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# Create trainer (builds encoder + optimizer + data loaders)
trainer = Trainer(config, model, tokenizer)

print(f"\n  Encoder on device: {next(trainer.encoder.parameters()).device}")
print(f"  Train dataset: {len(trainer.train_dataset)} samples")
print(f"  Val dataset:   {len(trainer.val_dataset)} samples")
print(f"  VRAM total:    {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 7. Train!

10,000 optimizer steps with cosine LR decay. Evals every 500 steps, checkpoints every 1,000. Best checkpoint saved by val_loss.

**Estimated time on A100 40GB: ~3–4 hours.**

In [ ]:
history = trainer.train()

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

# Load history (also available in memory from trainer)
history_path = f"{config.output_dir}/history.json"
try:
    with open(history_path) as f:
        history = json.load(f)
except FileNotFoundError:
    print("Using in-memory history")

steps = [h["step"] for h in history]
losses = [h["loss"] for h in history]
distills = [h["distill"] for h in history]
val_steps = [h["step"] for h in history if "val_loss" in h]
val_losses = [h["val_loss"] for h in history if "val_loss" in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
ax = axes[0]
ax.plot(steps, losses, alpha=0.3, color="steelblue", label="per-step")
# Smoothed
window = min(50, len(losses) // 5) if len(losses) > 10 else 1
if window > 1:
    import numpy as np
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    ax.plot(steps[window-1:], smoothed, color="navy", linewidth=2, label=f"smoothed ({window})")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss (KL Distillation)")
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss
ax = axes[1]
if val_steps:
    ax.plot(val_steps, val_losses, "o-", color="crimson", linewidth=2, markersize=6)
    ax.set_xlabel("Step")
    ax.set_ylabel("Val Loss")
    ax.set_title("Validation Loss")
    ax.grid(True, alpha=0.3)
    best_idx = val_losses.index(min(val_losses))
    ax.annotate(f"best: {val_losses[best_idx]:.4f}",
                xy=(val_steps[best_idx], val_losses[best_idx]),
                xytext=(10, 10), textcoords="offset points",
                fontweight="bold", color="green")
else:
    ax.text(0.5, 0.5, "No validation data yet", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig(f"{config.output_dir}/training_curves.png", dpi=150)
plt.show()
print(f"Final train loss: {losses[-1]:.4f}")
if val_losses:
    print(f"Best val loss:    {min(val_losses):.4f} (step {val_steps[val_losses.index(min(val_losses))]})")

## 9. Evaluate Best Checkpoint

Runs keyword hit ratio, perplexity, and per-query-type breakdown on the best encoder checkpoint.

In [ ]:
from experiments.level4_multibank.evaluate import (
    evaluate_encoder,
    print_eval_summary,
)
from experiments.level4_multibank.data.dataset import ProfileQueryDataset

# Load best encoder
ckpt_path = f"{config.output_dir}/encoder_best.pt"
checkpoint = torch.load(ckpt_path, map_location=config.device, weights_only=True)
trainer.encoder.load_state_dict(checkpoint["encoder_state_dict"])
print(f"Loaded checkpoint: step {checkpoint.get('step', '?')}, val_loss {checkpoint.get('val_loss', '?')}")

# Build val dataset
val_dataset = ProfileQueryDataset(
    data_path="experiments/level4_multibank/data/val.json",
    tokenizer=tokenizer,
    max_profile_tokens=config.encoder.max_profile_tokens,
    max_query_tokens=config.encoder.max_query_tokens,
)

# Run evaluation
eval_results = evaluate_encoder(
    config, model, tokenizer, trainer.encoder, val_dataset, max_samples=50
)
print_eval_summary(eval_results)

# Save
import json
eval_path = f"{config.output_dir}/eval_best.json"
with open(eval_path, "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print(f"\nResults saved to {eval_path}")

## 10. Sample Outputs — Gold vs Injection

Side-by-side comparison of gold (system prompt with facts) vs injection (encoder KV injection) for different query types.

In [ ]:
NUM_SAMPLES = 5

print("=" * 72)
print("  SAMPLE OUTPUTS")
print("=" * 72)

for i, sample in enumerate(eval_results["samples"][:NUM_SAMPLES]):
    print(f"\n{'─' * 72}")
    print(f"  [{i+1}] Query: {sample['query']}")
    print(f"      Types: {sample['relevant_types']}")
    print(f"      Keywords — inject: {sample['kw_inject']}/{sample['kw_total']}  gold: {sample['kw_gold']}/{sample['kw_total']}")
    print(f"      PPL      — inject: {sample['ppl_inject']:.1f}  gold: {sample['ppl_gold']:.1f}")
    print(f"      Gate     — mean: {sample['gate_mean']:.4f}  std: {sample['gate_std']:.4f}")
    print()
    print(f"  GOLD:   {sample['gold_text'][:200]}")
    print(f"  INJECT: {sample['inject_text'][:200]}")
print(f"\n{'─' * 72}")

## 11. Gate Specialization Heatmap

Visualize how the dynamic gate (sigmoid) values vary across layers and heads for different query types. This reveals whether the encoder learns to route different memory types through different layer groups.

In [ ]:
import numpy as np
from experiments.level4_multibank.data.dataset import ProfileQueryDataset, SUFFIX_TEMPLATE

# Collect gate values per query type
type_gates = {}  # {type_name: [gate_tensor(24,32), ...]}

for idx in range(min(len(val_dataset), 100)):
    sample = val_dataset[idx]
    raw = val_dataset.data[idx]
    types_tag = ", ".join(sorted(raw["relevant_types"]))

    profile_ids = sample["profile_ids"].unsqueeze(0).to(config.device)
    profile_mask = sample["profile_mask"].unsqueeze(0).to(config.device)
    query_ids = sample["query_ids"].unsqueeze(0).to(config.device)
    query_mask = sample["query_mask"].unsqueeze(0).to(config.device)

    with torch.no_grad():
        _, diag = trainer.encoder(
            profile_ids, profile_mask, query_ids, query_mask, return_diagnostics=True
        )

    gate = diag["gate_values"].squeeze(0).cpu().numpy()  # (24, 32)
    if types_tag not in type_gates:
        type_gates[types_tag] = []
    type_gates[types_tag].append(gate)

# Average gate per query type
top_types = sorted(type_gates.keys(), key=lambda t: len(type_gates[t]), reverse=True)[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, qtype in zip(axes.flatten(), top_types):
    avg_gate = np.mean(type_gates[qtype], axis=0)  # (24, 32)
    im = ax.imshow(avg_gate, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"types: {qtype}\n(n={len(type_gates[qtype])})", fontsize=9)
    ax.set_xlabel("KV Head")
    ax.set_ylabel("Layer")
    plt.colorbar(im, ax=ax, shrink=0.8)

# Hide unused axes
for ax in axes.flatten()[len(top_types):]:
    ax.set_visible(False)

fig.suptitle("Dynamic Gate Specialization per Query Type", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{config.output_dir}/gate_heatmap.png", dpi=150)
plt.show()
print("Gate heatmap saved.")

## 12. Working Memory Attention

Shows which memory bank slots the working memory module attends to. This reveals if the query-conditioned cross-attention learns to select from the right banks (episodic vs semantic vs procedural etc).

In [ ]:
# Working memory attention: (B, M=32, total_slots=72)
# Bank layout: episodic(32), semantic(16), procedural(8), emotional(8), prospective(8) = 72
bank_names = config.banks.bank_names
bank_sizes = config.banks.bank_sizes
bank_boundaries = np.cumsum([0] + bank_sizes)

# Collect WM attention per query type
type_wm_attn = {}

for idx in range(min(len(val_dataset), 100)):
    sample = val_dataset[idx]
    raw = val_dataset.data[idx]
    types_tag = ", ".join(sorted(raw["relevant_types"]))

    profile_ids = sample["profile_ids"].unsqueeze(0).to(config.device)
    profile_mask = sample["profile_mask"].unsqueeze(0).to(config.device)
    query_ids = sample["query_ids"].unsqueeze(0).to(config.device)
    query_mask = sample["query_mask"].unsqueeze(0).to(config.device)

    with torch.no_grad():
        _, diag = trainer.encoder(
            profile_ids, profile_mask, query_ids, query_mask, return_diagnostics=True
        )

    wm = diag["wm_attention"].squeeze(0).cpu().numpy()  # (32, 72)
    # Aggregate attention per bank
    bank_attn = []
    for i, name in enumerate(bank_names):
        start, end = bank_boundaries[i], bank_boundaries[i + 1]
        bank_attn.append(wm[:, start:end].sum(axis=1).mean())  # avg across output slots

    if types_tag not in type_wm_attn:
        type_wm_attn[types_tag] = []
    type_wm_attn[types_tag].append(bank_attn)

# Build bar chart
top_types = sorted(type_wm_attn.keys(), key=lambda t: len(type_wm_attn[t]), reverse=True)[:8]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(bank_names))
width = 0.8 / len(top_types)

for i, qtype in enumerate(top_types):
    means = np.mean(type_wm_attn[qtype], axis=0)
    bars = ax.bar(x + i * width, means, width, label=qtype[:30], alpha=0.8)

ax.set_xlabel("Memory Bank")
ax.set_ylabel("Avg Attention Mass")
ax.set_title("Working Memory Bank Attention by Query Type")
ax.set_xticks(x + width * len(top_types) / 2)
ax.set_xticklabels(bank_names)
ax.legend(fontsize=7, loc="upper right")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(f"{config.output_dir}/wm_attention.png", dpi=150)
plt.show()
print("Working memory attention chart saved.")

## 13. Download Checkpoint & Artifacts

Downloads the best encoder, training history, evaluation results, and visualizations.

In [ ]:
import shutil
from google.colab import files

# Zip everything
output_dir = config.output_dir
zip_name = "level4_multibank_outputs"
shutil.make_archive(zip_name, "zip", output_dir)

print(f"Contents of {output_dir}/:")
!ls -lh {output_dir}/

print(f"\nZip size: ", end="")
!ls -lh {zip_name}.zip

# Download
files.download(f"{zip_name}.zip")
print("\n✓ Download triggered — check your browser downloads.")